<a href="https://colab.research.google.com/github/MariamMohamed06/flyrank-ml-week1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MariamMohamed06/flyrank-ml-week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### My baseline rule

I will prioritize content pages that show strong search visibility together with signs that they may need review.

The baseline score will combine three observable signals:

- Search impressions: pages with more impressions have more visibility and therefore may have greater value if reviewed.
- Content age: older pages may deserve a freshness review.
- Average search position: pages that already have search visibility but are not consistently near the top may have an opportunity for review.

### Reason codes

- `high_visibility_old_page`: the page has strong impressions and is relatively old.
- `visible_position_opportunity`: the page has meaningful impressions but an average position that suggests room for improvement.

### Action label

The action label for both reason codes is `review`.

This is a prioritization rule, not a claim that changing a page will cause better performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import duckdb
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    PROVIDER credential_chain
)
""")

features_honest = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(scroll_events) AS scroll_events
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print(features_honest.shape)
features_honest.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 7)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,5.147402,NaN,NaN
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0,1.0,4.828125,NaN,NaN
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5.145765,NaN,NaN
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,4.909314,NaN,NaN
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,6.969536,NaN,NaN


In [ ]:
# Build the baseline score using only observed warehouse signals

baseline = features_honest.copy()

# Fill missing observed signals with 0
baseline["gsc_impressions"] = baseline["gsc_impressions"].fillna(0)
baseline["gsc_clicks"] = baseline["gsc_clicks"].fillna(0)
baseline["gsc_avg_position"] = baseline["gsc_avg_position"].fillna(0)
baseline["ga4_sessions"] = baseline["ga4_sessions"].fillna(0)
baseline["scroll_events"] = baseline["scroll_events"].fillna(0)

# Normalize signals to 0-100
def min_max(series):
    minimum = series.min()
    maximum = series.max()
    if maximum == minimum:
        return series * 0
    return (series - minimum) / (maximum - minimum) * 100

baseline["visibility_score"] = min_max(baseline["gsc_impressions"])

# For position, lower position number is better.
# We use the inverse so higher opportunity score means weaker position.
position_score = baseline["gsc_avg_position"]
baseline["position_opportunity_score"] = min_max(position_score)

baseline["engagement_score"] = min_max(
    baseline["ga4_sessions"] + baseline["scroll_events"]
)

# Final transparent baseline score
baseline["baseline_score"] = (
    0.50 * baseline["visibility_score"]
    + 0.30 * baseline["position_opportunity_score"]
    + 0.20 * baseline["engagement_score"]
)

# One reason code
baseline["reason_code"] = "visible_position_opportunity"

# One action label
baseline["action"] = "review"

# Rank pages
baseline = baseline.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = range(1, len(baseline) + 1)

# Write the required CSV
import os
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline.to_csv(output_path, index=False)

print(f"Saved {len(baseline):,} rows to {output_path}")

baseline.head(10)

Saved 331,437 rows to work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events,visibility_score,position_opportunity_score,engagement_score,baseline_score,reason_code,action,rank
0,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,2.383011,2730.0,444.0,100.000000,0.771201,100.000000,70.231360,visible_position_opportunity,review,1
1,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,32.766674,2603.0,365.0,31.529968,10.604102,93.509767,37.648168,visible_position_opportunity,review,2
2,client_08a6a72ff48e62c0,content_9e8c3b83214c180d,1.0,0.0,309.000000,0.0,0.0,0.000162,100.000000,0.000000,30.000081,visible_position_opportunity,review,3
3,client_3ffa76342f366962,content_06589faf15cc8488,1.0,0.0,297.000000,0.0,0.0,0.000162,96.116505,0.000000,28.835032,visible_position_opportunity,review,4
4,client_3ffa76342f366962,content_36cc2bda86ee726a,1.0,0.0,292.000000,0.0,0.0,0.000162,94.498382,0.000000,28.349596,visible_position_opportunity,review,5
5,client_08a6a72ff48e62c0,content_11187e07e5ee9f43,1.0,0.0,286.000000,0.0,0.0,0.000162,92.556634,0.000000,27.767071,visible_position_opportunity,review,6
6,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,15.008339,891.0,132.0,39.689106,4.857068,32.230624,27.747798,visible_position_opportunity,review,7
7,client_3ffa76342f366962,content_efce4eda2b012964,1.0,0.0,283.000000,0.0,0.0,0.000162,91.585761,0.000000,27.475809,visible_position_opportunity,review,8
8,client_3ffa76342f366962,content_0cec599cfeab8b7f,1.0,0.0,280.000000,0.0,0.0,0.000162,90.614887,0.000000,27.184547,visible_position_opportunity,review,9
9,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,2.854514,816.0,221.0,39.745011,0.923791,32.671708,26.683984,visible_position_opportunity,review,10


## 3. Top-20 review

1. **Rank 1 — content_eadb33b5df496f4a:** Action: review. Why: very high search impressions, strong traffic, and strong engagement make this a high-visibility page worth reviewing. Confidence: moderate. What would make it wrong: the page may already be performing well and need no change.

2. **Rank 2 — content_36e53e9c707674fc:** Action: review. Why: high impressions, sessions, and engagement with an average position around 33 suggest visible search opportunity. Confidence: moderate. What would make it wrong: the position may reflect a query mix that does not represent a meaningful optimization opportunity.

3. **Rank 3 — content_9e8c3b83214c180d:** Action: review. Why: the baseline score is driven mainly by the very weak average position. Confidence: low. What would make it wrong: only one impression was observed, so the position is not reliable evidence of a real opportunity.

4. **Rank 4 — content_06589faf15cc8488:** Action: review. Why: the score is driven by a very weak average position. Confidence: low. What would make it wrong: one impression provides almost no evidence of a meaningful search opportunity.

5. **Rank 5 — content_36cc2bda86ee726a:** Action: review. Why: the score is driven by the weak average position. Confidence: low. What would make it wrong: one impression and no clicks or sessions suggest insufficient evidence.

6. **Rank 6 — content_e8a52cf3d5988c07:** Action: review. Why: high impressions, clicks, sessions, and engagement combined with an average position around 15 make this a more credible review candidate. Confidence: moderate. What would make it wrong: the page may already be performing well for its important queries.

7. **Rank 7 — content_efce4eda2b012964:** Action: review. Why: the score is driven by a very weak average position. Confidence: low. What would make it wrong: one impression and no traffic make the signal too sparse.

8. **Rank 8 — content_0cec599cfeab8b7f:** Action: review. Why: the baseline ranks it because of its weak average position. Confidence: low. What would make it wrong: one impression is not enough evidence of a real opportunity.

9. **Rank 9 — content_ec2e0346994fb5a5:** Action: review. Why: high impressions, clicks, sessions, and engagement indicate substantial search visibility. Confidence: moderate. What would make it wrong: its average position is already strong, so a review may not produce additional value.

10. **Rank 10 — content_61b375eafb1d4c27:** Action: review. Why: the score is mainly caused by the weak average position. Confidence: low. What would make it wrong: one impression and no traffic are too sparse to support the recommendation.

11. **Rank 11 — content_fc468c5940d16ea3:** Action: review. Why: the baseline score reflects a very weak average position. Confidence: low. What would make it wrong: one impression does not establish a stable search pattern.

12. **Rank 12 — content_3758dd311e8033f7:** Action: review. Why: the page receives a high score because of its weak average position. Confidence: low. What would make it wrong: one impression and no clicks provide insufficient evidence.

13. **Rank 13 — content_d1b44ca865290810:** Action: review. Why: the baseline is driven by the weak average position. Confidence: low. What would make it wrong: one impression is too little evidence to justify prioritization.

14. **Rank 14 — content_8717cb37cf1fb1bc:** Action: review. Why: the page has a weak average position, although it has only two impressions. Confidence: low. What would make it wrong: the very small number of observations may make the position unstable.

15. **Rank 15 — content_3b427456fee115ac:** Action: review. Why: the score is driven mainly by the weak average position. Confidence: low. What would make it wrong: one impression and no traffic provide little evidence of a real opportunity.

16. **Rank 16 — content_b51957d7f4abe47e:** Action: review. Why: high impressions and sessions provide meaningful visibility, although the average position is around 29. Confidence: moderate. What would make it wrong: the low click volume may mean that the page's search visibility does not represent a useful improvement opportunity.

17. **Rank 17 — content_c62476fdb1c62107:** Action: review. Why: the score is driven by the weak average position. Confidence: low. What would make it wrong: one impression and no traffic are insufficient evidence.

18. **Rank 18 — content_f177a2fd99222df1:** Action: review. Why: the baseline ranks it because of its weak average position. Confidence: low. What would make it wrong: one impression provides almost no evidence of a stable opportunity.

19. **Rank 19 — content_33e5e7b9f9144e85:** Action: review. Why: the score is driven by its weak average position. Confidence: low. What would make it wrong: one impression and no traffic make the recommendation highly uncertain.

20. **Rank 20:** Action: review. Why: the baseline queue prioritizes this page based on the same visibility/position signals. Confidence: low. What would make it wrong: sparse observations or a context not captured by the baseline could make the recommendation inappropriate.

In [ ]:
top20 = baseline.head(20)[[
    "rank",
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "action",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events"
]]

top20

,rank,content_hash_id,baseline_score,reason_code,action,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,1,content_eadb33b5df496f4a,70.231360,visible_position_opportunity,review,617124.0,5668.0,2.383011,2730.0,444.0
1,2,content_36e53e9c707674fc,37.648168,visible_position_opportunity,review,194579.0,242.0,32.766674,2603.0,365.0
2,3,content_9e8c3b83214c180d,30.000081,visible_position_opportunity,review,1.0,0.0,309.000000,0.0,0.0
3,4,content_06589faf15cc8488,28.835032,visible_position_opportunity,review,1.0,0.0,297.000000,0.0,0.0
4,5,content_36cc2bda86ee726a,28.349596,visible_position_opportunity,review,1.0,0.0,292.000000,0.0,0.0
5,6,content_11187e07e5ee9f43,27.767071,visible_position_opportunity,review,1.0,0.0,286.000000,0.0,0.0
6,7,content_e8a52cf3d5988c07,27.747798,visible_position_opportunity,review,244931.0,669.0,15.008339,891.0,132.0
7,8,content_efce4eda2b012964,27.475809,visible_position_opportunity,review,1.0,0.0,283.000000,0.0,0.0
8,9,content_0cec599cfeab8b7f,27.184547,visible_position_opportunity,review,1.0,0.0,280.000000,0.0,0.0
9,10,content_ec2e0346994fb5a5,26.683984,visible_position_opportunity,review,245276.0,1480.0,2.854514,816.0,221.0


## 4. Weak picks + leakage check

Several weak picks in the top 20 are pages with only one or two impressions and no clicks or sessions. Their high ranking is mainly caused by extreme average-position values, so they should not be treated as strong recommendations.

This shows a limitation of the baseline: sparse observations can create misleading scores.

The baseline does not use product flags, future-window outcomes, or label-derived inputs. It uses only observed signals from the March 2026 feature window.

The ranked queue should therefore be treated as decision-support evidence for human review, not as an automatic decision.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.